# ScoreLens T3: synthetic matched-WT response diagnostic

This local walkthrough creates a synthetic T3 case in a temporary directory, runs the installed ScoreLens commands, and reads the resulting summary. It does not download data, contact a service, use Colab, or access hidden Challenge files. The result is a selected-subset synthetic diagnostic, not a biological or leaderboard result.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

work = Path(tempfile.mkdtemp(prefix='vec-scorelens-t3-'))
inputs = work / 'inputs'
report = work / 'report'
scorelens = Path(sys.executable).with_name('vec-scorelens')
scorelens_demo = Path(sys.executable).with_name('vec-scorelens-demo')
print(work)

In [ ]:
subprocess.run([
    str(scorelens_demo), '--task', 'T3', '--out', str(inputs), '--case', 'mixed'
], check=True)
subprocess.run([
    str(scorelens), '--task', 'T3',
    '--prediction', str(inputs / 'prediction.h5ad'),
    '--target', str(inputs / 'target.h5ad'),
    '--wt', str(inputs / 'reference.h5ad'),
    '--out', str(report),
], check=True)

In [ ]:
import re
from IPython.display import Image, Markdown, display
summary = (report / 'summary.md').read_text(encoding='utf-8')
pending = []
for line in summary.splitlines(keepends=True):
    image = re.fullmatch(r'!\[.*\]\((figures/[^)]+\.png)\)\n?', line)
    if image:
        if pending:
            display(Markdown(''.join(pending)))
            pending = []
        display(Image(filename=str(report / image.group(1))))
    else:
        pending.append(line)
if pending:
    display(Markdown(''.join(pending)))
sorted(path.name for path in report.iterdir())

## Interpret response values carefully

T3 compares matched-WT mean transformed expression: `target − WT` versus `prediction − WT`. These are not raw-count log fold changes. Per-gene residuals explain selected response patterns but do not decompose severity, establish a causal perturbation mechanism, or prescribe a model change. Spatial evidence has the same selected-scope and no-registration limits as T2.